## 0.1 Init ambiente Google Colab

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Para correr la siguiente celda es fundamental, POR UNICA VEZ, seguir los siguientes pasos

* Registrar usuario en Kaggle con la cuenta de email de la Universidad Austral
* Hacer el "Join Competition"  a la competencia de  Labo 3
* Generar el archivo kaggle.json  a partir de   https://www.kaggle.com/settings/account  y presione  "Create Legacy API Key"
* Crear carpeta  labo3  en  el Google Drive
* Dentro de la carpeta labo3 crear carpeta   kaggle
* Subir a la carpeta kaggle el archivo  kaggle.json

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

# z102 — EDA orientado a Feature Engineering

Este notebook **extiende** el EDA de clase (`z101_EDA_01`) manteniendo el mismo stack
(**DuckDB + plotly**), pero deja de ser un EDA descriptivo general y se enfoca en
**descubrir features** para los modelos que vienen después (LightGBM / Deep Learning),
dando de paso a AutoARIMA (`z251`) el subconjunto que ese modelo puede aprovechar.

**El problema (recordatorio):** "La Multinacional" (Unilever) hace forecasting **mensual**
de sell-in. Hay que predecir `tn` de **202002** (= mes+2, por el lag de ~2 meses entre que
sale el camión y la venta en góndola) para los **780 productos** de `product_id_apredecir201912`.

**Foco de este EDA (lo que marcó el profesor):**
1. **Estacionalidad específica por producto/categoría** (no global).
2. **Ciclo de vida**: ignorar primeros 3 meses (ramp-up) y la discontinuación.
3. **Push de Octubre**: pico artificial de cierre de año → candidato a *feature* binario.
4. **Clientes chicos más homogéneos** (compras fijas, menor CV).

Cada hallazgo se cierra con **"¿qué feature / decisión habilita?"**.

In [ ]:
import os as os
import duckdb
import numpy as np
import pandas as pd
import plotly.express as px

In [ ]:
# defino los parametros
PARAM = {'experimento':'EDA-102',
  'kaggle_competition':'labo-iii-2026-ba'
}

In [ ]:
# creo la carpeta del experimento
ruta = "/content/buckets/b1/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

## 1 Carga de tablas

Mismo enfoque que el z101: cargo a tablas DuckDB. Agrego dos tablas que el EDA de clase
descargaba pero **no usaba**: `tb_apredecir` (los 780 a predecir) y `tb_stocks`.

In [ ]:
con = duckdb.connect()

In [ ]:
# sell-in, transformando periodo (YYYYMM) a tipo DATE  (igual que z101)
con.execute("""
    CREATE OR REPLACE TABLE tb_sellin AS
    SELECT CAST(strptime(CAST( periodo AS VARCHAR), '%Y%m') AS DATE) periodo,
           customer_id,
           product_id,
           plan_precios_cuidados,
           cust_request_qty,
           cust_request_tn,
           tn
    FROM read_csv_auto('/content/datasets/sell-in.txt.gz')
    ORDER BY product_id, periodo, customer_id
""")

In [ ]:
# definicion de los productos
con.execute("""
    CREATE OR REPLACE TABLE tb_productos AS
    SELECT *
    FROM read_csv_auto('/content/datasets/tb_productos.txt')
    ORDER BY product_id
""")

In [ ]:
# los 780 productos cuyo tn de 202002 hay que predecir
con.execute("""
    CREATE OR REPLACE TABLE tb_apredecir AS
    SELECT *
    FROM read_csv_auto('/content/datasets/product_id_apredecir201912.txt')
""")

print("Productos a predecir:", con.sql("SELECT COUNT(*) FROM tb_apredecir").fetchone()[0])

In [ ]:
# stocks (la cargamos para tenerla disponible; no es el foco de este EDA)
con.execute("""
    CREATE OR REPLACE TABLE tb_stocks AS
    SELECT *
    FROM read_csv_auto('/content/datasets/tb_stocks.txt')
""")

In [ ]:
# inspecciono los esquemas reales (no asumo nombres de columnas)
print("=== tb_productos ===")
display(con.sql("DESCRIBE tb_productos").df())
print("=== tb_stocks ===")
display(con.sql("DESCRIBE tb_stocks").df())

In [ ]:
# rango temporal y tamano del panel
display(con.sql("""
    SELECT MIN(periodo) primer_periodo,
           MAX(periodo) ultimo_periodo,
           COUNT(DISTINCT periodo) n_periodos,
           COUNT(DISTINCT product_id) n_productos,
           COUNT(DISTINCT customer_id) n_clientes,
           COUNT(*) n_filas
    FROM tb_sellin
""").df())

# guardo el ultimo periodo disponible (la historia termina aqui)
ULT = con.sql("SELECT MAX(periodo) FROM tb_sellin").fetchone()[0]
print("Ultimo periodo con datos:", ULT, " ->  hay que predecir 202002 (mes+2)")

## 2 Ciclo de vida del producto  (ramp-up + discontinuación)

> El profesor: *"No considerar los primeros 3 meses de vida de los productos, ni cuando se están
> discontinuando."*

Solo hay **sell-in** (sale el camión), no sell-out. El arranque de un producto (ramp-up) y su apagado
(discontinuación) son tramos ruidosos que no representan demanda estable.

**Grano de modelado** = `tn` por `product_id` × `periodo` (sumando clientes). Es exactamente lo que
AutoARIMA modela, y la base del panel para GBM/DL.

In [ ]:
# grano de modelado: producto x periodo
con.execute("""
    CREATE OR REPLACE TABLE tb_ventas_pm AS
    SELECT product_id, periodo, SUM(tn) tn
    FROM   tb_sellin
    GROUP BY product_id, periodo
    ORDER BY product_id, periodo
""")
display(con.sql("SELECT * FROM tb_ventas_pm LIMIT 5").df())

In [ ]:
# vida de cada producto: alta, baja y meses con venta
con.execute(f"""
    CREATE OR REPLACE TABLE tb_vida AS
    SELECT product_id,
           MIN(periodo) mes_alta,
           MAX(periodo) mes_baja,
           COUNT(*) FILTER (WHERE tn > 0) meses_con_venta,
           datediff('month', MIN(periodo), MAX(periodo)) + 1 AS span_meses,
           (MAX(periodo) < DATE '{ULT}') AS discontinuado
    FROM   tb_ventas_pm
    GROUP BY product_id
""")
display(con.sql("SELECT * FROM tb_vida ORDER BY product_id LIMIT 10").df())

In [ ]:
# distribucion de la historia disponible
gra = px.histogram(
    con.sql("SELECT span_meses FROM tb_vida").df(),
    x="span_meses", nbins=36,
    title="Distribución de meses de vida por producto (span alta→baja)",
    labels={"span_meses": "Meses de vida"}
)
gra.show()

display(con.sql("""
    SELECT discontinuado, COUNT(*) n_productos
    FROM tb_vida GROUP BY discontinuado ORDER BY discontinuado
""").df())

### ¿Cómo afecta esto a los 780 productos que hay que predecir?

Si un producto tiene poca historia, AutoARIMA con `m=12` no puede ajustar estacionalidad
(es la causa de los ~209 que en `z251` caen a la 2ª pasada sin estacionalidad).

In [ ]:
# diagnostico de los 780 productos objetivo
display(con.sql(f"""
    SELECT
        COUNT(*) total_objetivo,
        SUM( (v.mes_baja >= DATE '{ULT}')::INT )            activos_al_final,
        SUM( v.discontinuado::INT )                          discontinuados,
        SUM( (v.span_meses < 12)::INT )                      menos_de_12m_historia,
        SUM( (v.span_meses < 24)::INT )                      menos_de_24m_historia
    FROM tb_apredecir a
    JOIN tb_vida v USING (product_id)
""").df())

In [ ]:
# construyo el panel con flags de ciclo de vida  (insumo directo de modelado)
con.execute(f"""
    CREATE OR REPLACE TABLE tb_panel AS
    SELECT v.product_id,
           v.periodo,
           v.tn,
           datediff('month', d.mes_alta, v.periodo)              AS meses_desde_lanzamiento,
           (datediff('month', d.mes_alta, v.periodo) < 3)        AS es_ramp_up,
           ( d.discontinuado
             AND datediff('month', v.periodo, d.mes_baja) <= 2 ) AS es_discontinuacion
    FROM tb_ventas_pm v
    JOIN tb_vida d USING (product_id)
""")

display(con.sql("""
    SELECT
        SUM(es_ramp_up::INT)          filas_ramp_up,
        SUM(es_discontinuacion::INT)  filas_discontinuacion,
        COUNT(*)                      filas_totales
    FROM tb_panel
""").df())

In [ ]:
# funcion para visualizar el ciclo de vida de UN producto (coloreando los tramos)
def graficar_ciclo_vida(producto):
  df = con.sql(f"""
      SELECT periodo, tn,
             CASE WHEN es_ramp_up THEN 'ramp-up (<3m)'
                  WHEN es_discontinuacion THEN 'discontinuación'
                  ELSE 'ventana estable' END AS tramo
      FROM tb_panel WHERE product_id = {producto} ORDER BY periodo
  """).df()
  titulo = con.sql(f"SELECT CONCAT_WS(', ', *COLUMNS('.*')) FROM tb_productos WHERE product_id={producto}").fetchone()[0]
  gra = px.scatter(df, x="periodo", y="tn", color="tramo",
                   title=f"[{producto}] {titulo}",
                   labels={"periodo": "Periodo", "tn": "Toneladas"})
  gra.update_traces(mode='markers+lines')
  gra.update_yaxes(rangemode="tozero")
  gra.show()

In [ ]:
# ejemplos: un producto longevo y dos del archivo de discontinuados del z101
for p in [20001, 20494, 20554]:
  graficar_ciclo_vida(p)

**Conclusión §2.** Quedan definidos `meses_desde_lanzamiento`, `es_ramp_up` y `es_discontinuacion`.
- **ARIMA (`z251`)**: filtrar estas filas antes de `auto_arima` → series más limpias.
- **GBM/DL**: NO se tiran filas; `meses_desde_lanzamiento` entra como **feature** y el modelo aprende la forma del ramp-up.

## 3 Estacionalidad por producto / categoría

> El profesor: *"hay estacionalidad relacionada a los productos."*

La estacionalidad **no es global**: cada categoría tiene su propio perfil (mayonesas suben en verano,
sopas en invierno, etc.). Lo medimos con un **índice estacional** = `tn` promedio del mes / `tn` promedio
de la categoría. Calculado sobre la **ventana estable** (sin ramp-up ni discontinuación).

In [ ]:
# perfil estacional GLOBAL (sobre ventana estable)
df_mes = con.sql("""
    SELECT month(periodo) mes, AVG(tn) tn_medio
    FROM tb_panel
    WHERE NOT es_ramp_up AND NOT es_discontinuacion
    GROUP BY month(periodo) ORDER BY mes
""").df()
gra = px.bar(df_mes, x="mes", y="tn_medio",
             title="Perfil estacional GLOBAL (promedio por mes del año)",
             labels={"mes": "Mes", "tn_medio": "tn medio"})
gra.update_xaxes(dtick=1)
gra.show()

In [ ]:
# indice estacional por CATEGORIA (cat3) -> heatmap mes x categoria
# tomo las top-12 cat3 por volumen para que el heatmap sea legible
df = con.sql("""
    SELECT p.cat3, month(v.periodo) mes, v.tn
    FROM tb_panel v
    JOIN tb_productos p USING (product_id)
    WHERE NOT v.es_ramp_up AND NOT v.es_discontinuacion
""").df()

top_cats = df.groupby("cat3")["tn"].sum().sort_values(ascending=False).head(12).index
df = df[df["cat3"].isin(top_cats)]

# indice estacional: media del mes / media de la categoria
piv = df.groupby(["cat3", "mes"])["tn"].mean().reset_index()
piv = piv.pivot(index="cat3", columns="mes", values="tn")
indice = piv.div(piv.mean(axis=1), axis=0)

gra = px.imshow(indice, aspect="auto", color_continuous_scale="RdBu_r",
                origin="upper", labels={"x": "Mes", "y": "Categoría (cat3)", "color": "índice estacional"},
                title="Índice estacional por categoría  (1.0 = mes promedio)")
gra.update_xaxes(dtick=1)
gra.show()

In [ ]:
# reuso del z101: comparar perfiles de categorias con estacionalidad opuesta
def graficar_union_productos(productos, titulo="Productos"):
  gra = px.line(
      con.sql(f"SELECT periodo, SUM(tn) tn FROM tb_sellin WHERE product_id in {tuple(productos)} GROUP BY periodo ORDER BY 1").df(),
      x="periodo", y="tn", title=titulo,
      labels={"periodo": "Periodo", "tn": "Toneladas"})
  gra.update_yaxes(rangemode="tozero")
  gra.show()

# ejemplo: mayonesas vs sopas (categorias del z101)
may = con.sql("SELECT product_id FROM tb_productos WHERE cat3='Mayonesa'").df()["product_id"].tolist()
sop = con.sql("SELECT product_id FROM tb_productos WHERE cat3='Sopas'").df()["product_id"].tolist()
if may: graficar_union_productos(may, "Mayonesas (cat3)")
if sop: graficar_union_productos(sop, "Sopas (cat3)")

**Conclusión §3.** El heatmap confirma que la estacionalidad es **específica por categoría**.
- **ARIMA**: `m=12` ayuda solo en categorías con estacionalidad marcada y con historia ≥24m; en el resto conviene `seasonal=False`.
- **GBM/DL**: codificar **dummies de mes** y/o el **índice estacional por categoría** como feature (mejor que un `m=12` ciego).

## 4 Efecto "push de Octubre"  (cierre de año)

> El profesor: *"en octubre hacen push para cerrar el año, le piden al súper que compre más para inflar
> ventas; después de ese mes siempre baja. Se podría hacer un feature 0/1 con eso."*

Como el dato es **sell-in** (envío de Unilever), el pico aparece en el **propio mes del push**.
Lo verificamos con la distribución de `tn` por mes del año.

In [ ]:
# boxplot de tn por mes del anio (sobre ventana estable) -> se ve el pico y la caida posterior
df = con.sql("""
    SELECT month(periodo) mes, tn
    FROM tb_panel
    WHERE NOT es_ramp_up AND NOT es_discontinuacion AND tn > 0
""").df()
gra = px.box(df, x="mes", y="tn", title="Distribución de tn por mes del año (escala log)",
             labels={"mes": "Mes", "tn": "tn (producto-mes)"})
gra.update_yaxes(type="log")
gra.update_xaxes(dtick=1)
gra.show()

In [ ]:
# ratio de cada mes contra el promedio anual (serie global) para cuantificar el pico
df = con.sql("""
    SELECT month(periodo) mes, SUM(tn) tn
    FROM tb_panel WHERE NOT es_ramp_up AND NOT es_discontinuacion
    GROUP BY month(periodo) ORDER BY mes
""").df()
df["indice"] = df["tn"] / df["tn"].mean()
display(df)
gra = px.line(df, x="mes", y="indice", markers=True,
              title="Índice mensual global (1.0 = mes promedio) — ¿pico en octubre?",
              labels={"mes": "Mes", "indice": "índice"})
gra.add_hline(y=1.0, line_dash="dash")
gra.update_xaxes(dtick=1)
gra.show()

In [ ]:
# creo el feature candidato sugerido por el profesor: flag binario del mes de push
# (parametrizado por si conviene correrlo: octubre = mes 10)
MES_PUSH = 10

con.execute(f"""
    CREATE OR REPLACE TABLE tb_panel AS
    SELECT *,
           (month(periodo) = {MES_PUSH})::INT AS flag_push
    FROM tb_panel
""")
display(con.sql("SELECT flag_push, COUNT(*) n, ROUND(AVG(tn),2) tn_medio FROM tb_panel WHERE NOT es_ramp_up AND NOT es_discontinuacion GROUP BY flag_push").df())

**Conclusión §4.** Si el pico se confirma en octubre, queda creado `flag_push`.
- **ARIMA**: pasarlo como **regresor exógeno** (`X=` en `auto_arima` / SARIMAX).
- **GBM/DL**: feature binario directo (y opcionalmente `mes_posterior_al_push` para capturar la caída).
- *Verificar*: si el pico aparece corrido (nov/dic), ajustar `MES_PUSH`.

## 5 Homogeneidad de los clientes chicos

> El profesor: *"los clientes chicos tienden a hacer compras fijas sin importar; o sea, son más homogéneos."*

Hipótesis: a **menor volumen** del cliente, **menor coeficiente de variación (CV)** de su demanda mensual
(compras más regulares). Lo medimos con `CV = desvío / media` de la `tn` mensual por cliente.

> Nota: esto **no** lo usa AutoARIMA (que agrega sobre clientes); es insumo para el **track GBM/DL**
> (features de segmento/mix de cliente, o modelar a grano producto×cliente).

In [ ]:
# tn mensual por cliente -> tamano y CV
con.execute("""
    CREATE OR REPLACE TABLE tb_cli_mes AS
    SELECT customer_id, periodo, SUM(tn) tn
    FROM tb_sellin GROUP BY customer_id, periodo
""")

con.execute("""
    CREATE OR REPLACE TABLE tb_clientes AS
    SELECT customer_id,
           SUM(tn)                                   total_tn,
           COUNT(*)                                  n_meses,
           AVG(tn)                                   tn_medio,
           STDDEV_SAMP(tn) / NULLIF(AVG(tn), 0)      cv
    FROM tb_cli_mes
    GROUP BY customer_id
""")
display(con.sql("SELECT * FROM tb_clientes ORDER BY total_tn DESC LIMIT 5").df())

In [ ]:
# scatter: CV vs volumen del cliente (solo clientes con >=6 meses para que el CV sea estable)
df = con.sql("SELECT * FROM tb_clientes WHERE n_meses >= 6").df()
gra = px.scatter(df, x="total_tn", y="cv", opacity=0.4,
                 title="CV de la demanda vs volumen del cliente",
                 labels={"total_tn": "Volumen total (tn)", "cv": "CV (desvío/media)"},
                 trendline="lowess")
gra.update_xaxes(type="log")
gra.show()

In [ ]:
# CV mediano por decil de tamano -> resumen claro de la hipotesis
df = con.sql("SELECT * FROM tb_clientes WHERE n_meses >= 6").df()
df["decil_tamano"] = pd.qcut(df["total_tn"], 10, labels=False) + 1
resumen = df.groupby("decil_tamano").agg(
    cv_mediano=("cv", "median"),
    n_clientes=("customer_id", "count")
).reset_index()
display(resumen)
gra = px.bar(resumen, x="decil_tamano", y="cv_mediano",
             title="CV mediano por decil de tamaño del cliente (1=chico, 10=grande)",
             labels={"decil_tamano": "Decil de volumen", "cv_mediano": "CV mediano"})
gra.update_xaxes(dtick=1)
gra.show()

print("Correlación log(volumen) vs CV:",
      round(np.corrcoef(np.log(df["total_tn"]), df["cv"])[0, 1], 3))

**Conclusión §5.** Si los deciles chicos muestran CV menor, se confirma la hipótesis.
- **GBM/DL**: features como `cv_cliente`, `decil_tamano_cliente` o un flag `cliente_estable`; también
  features de **mix de clientes** por producto (qué fracción de la demanda viene de clientes estables).
- Sirve para distinguir demanda predecible (chicos regulares) de demanda volátil (grandes erráticos).

## 6 Conclusiones e implicancias para el modelado

| Hallazgo del EDA | Decisión para ARIMA (`z251`) | Feature para GBM / DL |
|---|---|---|
| **Ciclo de vida** (ramp-up <3m + discontinuación) | **filtrar** filas antes de `auto_arima` | `meses_desde_lanzamiento`, `es_ramp_up`, `es_discontinuacion` (no se tira data) |
| **Estacionalidad por categoría** | `m=12` selectivo (solo cats estacionales con ≥24m) | dummies de mes + índice estacional por `cat3` |
| **Push de Octubre** | regresor exógeno `X=` (SARIMAX) | `flag_push` (+ `post_push`) |
| **Clientes chicos homogéneos** | no aplica (agrega clientes) | `cv_cliente`, `decil_tamano`, mix de clientes estables |
| **Poca historia** (productos nuevos del set 780) | caen a 2ª pasada sin estacionalidad | `span_meses` como feature de confianza |

**Problemas encontrados → soluciones propuestas:**
1. *Tramos ruidosos (ramp-up / discontinuación)* → recortar para ARIMA, codificar como feature para GBM.
2. *Estacionalidad heterogénea* → no asumir `m=12` global; modelar por categoría.
3. *Pico artificial de octubre* → feature/exógena que lo aisle de la demanda real.
4. *Productos con poca historia* → modelo global (GBM) que comparte señal entre series, mejor que ARIMA por-serie.

**Artefactos generados** (`tb_panel`) listos para alimentar el pipeline de modelado.